# PSTU demo: per-secret-type task-vector unlearning

Companion notebook for *"Not All Secrets Are Equal: Type-Aware Unlearning for
Language Model Secret Removal"*.

PSTU is **training-free**. Given an infected model $\theta$ and a clean
reference $\theta_0$, it forms the task vector $\tau = \theta - \theta_0$ and
subtracts it back with **per-type, per-layer-group** strengths derived from
gradient saliency (optionally after `PSTU-Trim` denoising):

$$\theta^* = \theta - \Lambda \odot \tau'.$$

This notebook is a small **CPU-only** demonstration of the core mechanics on
random tensors. It does **not** download models or reproduce paper numbers —
use the scripts in `scripts/` for the full experiments.

In [ ]:
import torch
from pstu.method import apply_pstu, _compute_trim_threshold
from pstu.utils import detect_num_layers, param_group

torch.manual_seed(0)

# A tiny 4-layer Pythia-style parameter set (names only matter for grouping).
names = [
    "gpt_neox.embed_in.weight",
    "gpt_neox.layers.0.attention.dense.weight",
    "gpt_neox.layers.1.mlp.dense_4h_to_h.weight",
    "gpt_neox.layers.2.attention.dense.weight",
    "gpt_neox.layers.3.mlp.dense_4h_to_h.weight",
    "embed_out.weight",
]
clean = {n: torch.zeros(8, 8) for n in names}            # clean reference theta_0
infected = {n: torch.randn(8, 8) * 0.1 for n in names}   # infected theta
task_vectors = {n: infected[n] - clean[n] for n in names}
saliency = {n: i / len(names) for i, n in enumerate(names)}

n_layers = detect_num_layers(clean)
print("detected layers:", n_layers)

## Parameter grouping

Each parameter is mapped to a scaling group (`embed`, `head`, or `g{i}` for
layer block `i`). PSTU assigns an independent strength per group.

In [ ]:
for n in names:
    print(f"{param_group(n, n_layers, group_size=2):>6}  <-  {n}")

## Adaptive subtraction

With all group strengths set to 1 and no saliency boost, subtracting the full
task vector recovers the clean reference. `PSTU-Trim` keeps only the
high-magnitude entries, so a fraction of the task vector is left in place.

In [ ]:
alphas = {"embed": 1.0, "head": 1.0, "g0": 1.0, "g1": 1.0}

# Full subtraction (no trim) reconstructs the clean reference.
unlearned = apply_pstu(infected, clean, task_vectors, saliency, alphas,
                       saliency_boost=0.0, n_layers=n_layers, group_size=2,
                       trim_fraction=0.0, device="cpu")
max_gap = max((unlearned[n] - clean[n]).abs().max().item() for n in names)
print(f"max |theta* - theta_0| without trim: {max_gap:.2e}")

# PSTU-Trim: drop the smallest 50% of task-vector entries before subtracting.
thr = _compute_trim_threshold(task_vectors, 0.5, "cpu")
trimmed = apply_pstu(infected, clean, task_vectors, saliency, alphas,
                     saliency_boost=0.0, n_layers=n_layers, group_size=2,
                     trim_fraction=0.5, device="cpu")
residual = sum((trimmed[n] - clean[n]).abs().sum().item() for n in names)
print(f"trim threshold (50% quantile): {thr:.4f}")
print(f"residual left in place after trim: {residual:.4f}")

## Running the real experiments

The full pipeline uses real checkpoints and the Carlini exposure / WikiText-2
perplexity metrics. See the `scripts/` directory; for example:

```bash
# 1) infect a base model on the synthetic secrets
MODEL_SIZE=1.4b EPOCHS=4 python scripts/infect_model.py

# 2) run PSTU with the two-phase Optuna search (Tables 1-2)
python scripts/run_pstu.py --model pythia-1.4b --n-trials 500

# 3) evaluate a checkpoint (memorized count, exposure, perplexity)
python scripts/evaluate_model.py \
    --model-path results/pstu_comprehensive/pythia-1.4b_best_model_final \
    --clean-model EleutherAI/pythia-1.4b
```

Data under `data/` is synthetic and released under CC-BY-4.0 (see
`data/DATACARD.md`); code is Apache-2.0.